# 01 · Embeddings e índice de recuperación — piloto local
 `Qwen/Qwen3-Embedding-0.6B`.

Esta libreta utiliza el catálogo ya auditado y construye **un vector por `article_id`**, sin modificar artículos ni grafos. El objetivo es comprobar la vectorización y correspondencia de los resultados **antes de comparar** BM25, BGE-M3 y Qwen3-Embedding-4B.

**Entradas originales:** `06_grafos/catalogo_articulos.csv`, `07_RAG/auditoria_catalogo.json`, `07_RAG/consultas_para_revision.csv`.

No se calculan métricas de calidad con consultas pendientes de revisión humana. Los artículos con resúmenes idénticos mantienen sus identificadores distintos.

## 1. Verificar los datos y preparar el orden de los artículos

In [5]:
import csv
import hashlib
import json
import sys
from pathlib import Path
from collections import Counter

# La libreta puede abrirse desde 07_RAG o la raíz del repositorio.
carpetas = [Path.cwd(), Path.cwd().parent]
REPO = next((p.resolve() for p in carpetas
             if (p / "06_grafos" / "catalogo_articulos.csv").is_file()
             and (p / "07_RAG" / "auditoria_catalogo.json").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Abre el notebook desde 07_RAG o Tesis_Multimodelo.")

BASE = REPO / "07_RAG"
RUTA_CATALOGO = REPO / "06_grafos" / "catalogo_articulos.csv"
RUTA_AUDITORIA = BASE / "auditoria_catalogo.json"
RUTA_CONSULTAS = BASE / "consultas_para_revision.csv"

sha = lambda ruta: hashlib.sha256(ruta.read_bytes()).hexdigest()
auditoria = json.loads(RUTA_AUDITORIA.read_text(encoding="utf-8"))
SHA_CATALOGO = sha(RUTA_CATALOGO)
assert SHA_CATALOGO == auditoria["fuentes"]["catalogo"]["sha256_bytes"],     "El catálogo cambió desde la auditoría. No continuar antes de revisar."

with RUTA_CATALOGO.open(encoding="utf-8-sig", newline="") as f:
    articulos = list(csv.DictReader(f))

article_ids = [r["article_id"] for r in articulos]
resumenes = [r["abstract"] for r in articulos]
assert len(articulos) == 519 == len(set(article_ids))
assert all(t.strip() for t in resumenes)
assert len(set(resumenes)) == 514  # No se fusionan los cinco pares.

if RUTA_CONSULTAS.is_file():
    with RUTA_CONSULTAS.open(encoding="utf-8-sig", newline="") as f:
        consultas = list(csv.DictReader(f))
    estados = dict(Counter(c["revision_humana"] for c in consultas))
    print("Consultas:", len(consultas), "| revisión humana:", estados)
else:
    consultas = []
    print("No se encontró el CSV de consultas; el índice aún puede construirse.")

print("Python:", sys.version.split()[0])
print("Catálogo verificado, SHA-256:", SHA_CATALOGO)
print("Artículos:", len(article_ids), "| Resúmenes distintos:", len(set(resumenes)))
print("Ejemplos de article_id:", article_ids[:5])
print("Ruta de salida:", BASE)
print("Esta celda NO ejecuta ningún modelo.")

Consultas: 24 | revisión humana: {'pendiente': 24}
Python: 3.11.16
Catálogo verificado, SHA-256: 74040d185b762721c29521fdc3a270740569f009b7ae40dbce654366b576598b
Artículos: 519 | Resúmenes distintos: 514
Ejemplos de article_id: ['1', '2', '3', '4', '5']
Ruta de salida: C:\Users\hazar\Documents\GitHub\Tesis_Multimodelo\07_RAG
Esta celda NO ejecuta ningún modelo.


## 2. Comprobar las bibliotecas

**Ejecutar después de verificar los CSV.** 

```python
%pip install "transformers>=4.51.0" "sentence-transformers>=2.7.0" faiss-cpu
```

```python
%pip install torch==2.11.0 --index-url https://download.pytorch.org/whl/cu128
```

In [6]:
try:
    import numpy as np
    import torch
    import faiss
    import sentence_transformers
    import transformers
    from sentence_transformers import SentenceTransformer
except ImportError as exc:
    raise ImportError("Falta una biblioteca. Lee las instrucciones de instalación de la celda anterior.") from exc

print("numpy:", np.__version__)
print("torch:", torch.__version__, "| CUDA disponible:", torch.cuda.is_available())
print("sentence-transformers:", sentence_transformers.__version__)
print("transformers:", transformers.__version__)
print("faiss:", getattr(faiss, "__version__", "no declarada"))
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("AVISO: PyTorch utilizará CPU; no confundir con Ollama, que usa su propio motor.")
print("Bibliotecas listas; todavía no se han descargado modelos ni creado embeddings.")

numpy: 2.3.5
torch: 2.11.0+cu128 | CUDA disponible: True
sentence-transformers: 6.2.0
transformers: 5.22.2
faiss: 1.15.1
GPU: NVIDIA GeForce RTX 4070 SUPER
Bibliotecas listas; todavía no se han descargado modelos ni creado embeddings.


## 3. Codificar los 519 resúmenes y crear FAISS exacto

**Por seguridad, comienza con `EJECUTAR = False`.** Solo al activarlo se cargará el modelo, se generarán vectores y se guardarán dos archivos. Se usa la preparación de documentos indicada en la ficha de Qwen, sin prefijo; la consulta utiliza `prompt_name="query"`. El índice usa producto interno sobre vectores unitarios para recuperar por coseno.

**No vuelve a entrenar el modelo**, no usa los artículos orientadores para guiar el índice y no modifica el catálogo.

In [7]:
EJECUTAR = True 
MODELO = "Qwen/Qwen3-Embedding-0.6B"
VECTORES = BASE / "qwen3_0_6b_vectores_v1.npz"
INDICE = BASE / "qwen3_0_6b_indice_v1.faiss"

if not EJECUTAR:
    print("Preparado. No se ha descargado el modelo ni creado el índice.")
else:
    if VECTORES.exists() or INDICE.exists():
        raise FileExistsError("Los archivos del piloto ya existen. No se sobrescriben automáticamente.")

    from datetime import datetime, timezone
    from time import perf_counter
    import gc

    dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
    inicio = perf_counter()
    codificador = SentenceTransformer(MODELO, device=dispositivo)
    t_carga = perf_counter() - inicio
    print("Modelo cargado en", dispositivo, "| Tiempo de carga:", round(t_carga, 2), "s")

    # Conservar literalmente los resúmenes; un embedding por identificador de artículo.
    inicio = perf_counter()
    vectores = codificador.encode(
        resumenes, batch_size=4, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True
    )
    t_vectorizacion = perf_counter() - inicio
    # La conversión a float32 no vuelve a normalizar los vectores.
    # Registrar la precisión real y comprobar la salida antes de indexar.
    dtype_salida_encode = str(np.asarray(vectores).dtype)
    dtype_modelo = str(next(codificador.parameters()).dtype)
    vectores = np.array(vectores, dtype=np.float32, order="C", copy=True)
    if (vectores.ndim != 2 or vectores.shape[0] != len(article_ids)
            or vectores.shape[1] == 0 or not np.isfinite(vectores).all()):
        raise ValueError("Cantidad, forma o valores de los embeddings incorrectos.")

    normas_antes = np.linalg.norm(vectores, axis=1)
    if not np.isfinite(normas_antes).all() or np.any(normas_antes <= 0):
        raise ValueError("Hay vectores nulos o normas no finitas; no se pueden usar para coseno.")
    print("Precisión del modelo:", dtype_modelo, "| Salida de encode:", dtype_salida_encode)
    print("Normas ANTES (mín./máx.):", float(normas_antes.min()), float(normas_antes.max()))
    print("Desviación máxima ANTES:", float(np.max(np.abs(normas_antes - 1))))

    # Ajuste L2 final sobre float32, antes de usar producto interno como coseno.
    # No se entrena el modelo ni se cambia el texto: cada fila se divide por su norma.
    inicio_normalizacion = perf_counter()
    faiss.normalize_L2(vectores)
    t_normalizacion = perf_counter() - inicio_normalizacion
    normas = np.linalg.norm(vectores, axis=1)
    if (not np.isfinite(vectores).all()
            or not np.allclose(normas, 1.0, rtol=0.0, atol=1e-5)):
        raise ValueError("Falló la normalización L2 en float32; no se guarda el índice.")
    print("Normas DESPUÉS (mín./máx.):", float(normas.min()), float(normas.max()))
    print("Desviación máxima DESPUÉS:", float(np.max(np.abs(normas - 1))))

    indice = faiss.IndexFlatIP(vectores.shape[1])
    indice.add(vectores)
    assert indice.ntotal == len(article_ids)

    # Guardar la procedencia, versiones y orden de identificadores dentro del NPZ.
    mod = codificador._first_module()
    revision = getattr(getattr(getattr(mod, "auto_model", None), "config", None), "_commit_hash", None)
    np.savez_compressed(
        VECTORES, vectores=vectores, article_ids=np.asarray(article_ids),
        modelo=np.asarray(MODELO), modelo_revision=np.asarray(revision or "no_registrada"),
        sha256_catalogo=np.asarray(SHA_CATALOGO),
        sha256_auditoria=np.asarray(sha(RUTA_AUDITORIA)),
        fecha_utc=np.asarray(datetime.now(timezone.utc).isoformat()),
        dispositivo=np.asarray(dispositivo),
        version_codigo=np.asarray("EMBEDDINGS_INDICE_RAG_V1_FIX_L2_FP32"),
        dtype_modelo=np.asarray(dtype_modelo),
        dtype_salida_encode=np.asarray(dtype_salida_encode),
        normalizacion_final=np.asarray("faiss.normalize_L2 sobre float32; rtol=0; atol=1e-5"),
        normas_antes_min=np.asarray(float(normas_antes.min())),
        normas_antes_max=np.asarray(float(normas_antes.max())),
        desviacion_norma_antes_max=np.asarray(float(np.max(np.abs(normas_antes - 1)))),
        normas_despues_min=np.asarray(float(normas.min())),
        normas_despues_max=np.asarray(float(normas.max())),
        desviacion_norma_despues_max=np.asarray(float(np.max(np.abs(normas - 1)))),
        tiempo_normalizacion_s=np.asarray(t_normalizacion),
        campo=np.asarray("abstract"),
        tiempo_carga_s=np.asarray(t_carga),
        tiempo_vectorizacion_s=np.asarray(t_vectorizacion),
        python=np.asarray(sys.version.split()[0]),
        torch_version=np.asarray(torch.__version__),
        sentence_transformers_version=np.asarray(sentence_transformers.__version__),
        transformers_version=np.asarray(transformers.__version__),
        faiss_version=np.asarray(getattr(faiss, "__version__", "no_registrada"))
    )
    faiss.write_index(indice, str(INDICE))

    # Volver a leer y comprobar que los archivos conservan el orden correcto.
    with np.load(VECTORES, allow_pickle=False) as copia:
        assert copia["article_ids"].tolist() == article_ids
        assert copia["sha256_catalogo"].item() == SHA_CATALOGO
        assert np.array_equal(copia["vectores"], vectores)
    indice_copia = faiss.read_index(str(INDICE))
    assert indice_copia.ntotal == len(article_ids)
    print("Artículos indexados:", indice_copia.ntotal,
          "| Dimensión:", indice_copia.d,
          "| Vectorización:", round(t_vectorizacion, 2), "s")
    print("Guardados:", VECTORES.name, "y", INDICE.name)
    print("SHA-256 NPZ:", sha(VECTORES))
    print("SHA-256 FAISS:", sha(INDICE))
    print("No se han calculado métricas de recuperación.")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Modelo cargado en cuda | Tiempo de carga: 2.73 s


Batches:   0%|          | 0/130 [00:00<?, ?it/s]

Precisión del modelo: torch.bfloat16 | Salida de encode: float32
Normas ANTES (mín./máx.): 0.998051643371582 1.0037858486175537
Desviación máxima ANTES: 0.003785848617553711
Normas DESPUÉS (mín./máx.): 0.9999999403953552 1.000000238418579
Desviación máxima DESPUÉS: 2.384185791015625e-07
Artículos indexados: 519 | Dimensión: 1024 | Vectorización: 5.9 s
Guardados: qwen3_0_6b_vectores_v1.npz y qwen3_0_6b_indice_v1.faiss
SHA-256 NPZ: a8312779de2186fc3fff56db68d50bd44650ecbd347b557e1c7d3fa59a58234d
SHA-256 FAISS: 54a9ef8f854d71441f841c8ba59c900327a68c3ac813de9372838ae44f128615
No se han calculado métricas de recuperación.


## 4. Comprobación de búsqueda (sin evaluación de calidad)

Al crear el índice, esta celda permite verificar de manera ilustrativa que se puede recuperar el identificador correcto. Usa `encode(..., prompt_name="query")` para preparar la consulta de forma diferente al documento. **No interpreta los artículos orientadores como etiquetas definitivas ni informa precisión, recall o nDCG.**

In [8]:
# La demostración no modifica los dos archivos guardados.
VECTORES = BASE / "qwen3_0_6b_vectores_v1.npz"
INDICE = BASE / "qwen3_0_6b_indice_v1.faiss"
if not VECTORES.is_file() or not INDICE.is_file():
    print("Primero ejecuta la celda 3 con EJECUTAR=True.")
elif "codificador" not in globals():
    print("El índice está guardado. Falta cargar el codificador para esta demostración.")
    print('Carga únicamente: codificador = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device="cuda" if torch.cuda.is_available() else "cpu")')
else:
    # Comprobar la correspondencia con los identificadores guardados.
    with np.load(VECTORES, allow_pickle=False) as archivo:
        ids_guardados = archivo["article_ids"].tolist()
        if ids_guardados != article_ids or archivo["sha256_catalogo"].item() != SHA_CATALOGO:
            raise ValueError("El índice y el catálogo no corresponden al mismo orden o versión.")
    indice_busqueda = faiss.read_index(str(INDICE))
    if indice_busqueda.ntotal != len(ids_guardados):
        raise ValueError("El índice no tiene la cantidad esperada de artículos.")

    q = (consultas[0]["consulta_propuesta"] if consultas else
         "¿Qué artículos estudian documentación técnica mediante RAG?")
    vq = codificador.encode([q], prompt_name="query", normalize_embeddings=True, convert_to_numpy=True)
    vq = np.array(vq, dtype=np.float32, order="C", copy=True)
    if vq.ndim != 2 or vq.shape != (1, indice_busqueda.d) or not np.isfinite(vq).all():
        raise ValueError("El vector de consulta no es válido o su dimensión no coincide con el índice.")
    norma_q = np.linalg.norm(vq, axis=1)
    if not np.isfinite(norma_q).all() or np.any(norma_q <= 0):
        raise ValueError("La consulta produjo un vector nulo o una norma no finita.")

    # Mismo ajuste final que para los documentos: producto interno = coseno.
    faiss.normalize_L2(vq)
    if not np.allclose(np.linalg.norm(vq, axis=1), 1.0, rtol=0.0, atol=1e-5):
        raise ValueError("La consulta no quedó normalizada.")
    posiciones = indice_busqueda.search(vq, min(5, indice_busqueda.ntotal))[1][0].tolist()
    print("Consulta de ejemplo:", q)
    print("Cinco resultados, SIN calificación de pertinencia:")
    por_id = {r["article_id"]: r for r in articulos}
    for i in posiciones:
        if i >= 0:
            aid = ids_guardados[i]
            print(f"article_id={aid} | {por_id[aid]['titulo'][:95]}")

Consulta de ejemplo: ¿Qué trabajos permiten consultar documentación técnica de un sistema mediante preguntas en lenguaje natural y fundamentan la respuesta en material recuperado?
Cinco resultados, SIN calificación de pertinencia:
article_id=117 | A retrieval-augmented-generation pipeline to help users query system-provided documentation
article_id=184 | GIL-IIMAS UNAM at SemEval-2025 Task 3: MeSSI: A Multimodule System to Detect Hallucinated Segme
article_id=175 | Exploring the Behavior and Performance of Large Language Models: Can LLMs Infer Answers to Ques
article_id=253 | Spatial Analysis of Student Feedback on Learning Management Systems in Mathematics: Leveraging 
article_id=295 | Criteria for Assessing Digital Preservation Systems
